In [1]:
import torch
import torch.nn as nn 
from torch.nn import functional as F

block_size = 8
batch_size = 4
max_iters = 10000
learning_rate = 3e-4

In [7]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as file:
    text = file.read()
    chars = sorted(set(text))
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [16]:
string_to_int = {char:i for i,char in enumerate(chars)}
int_to_string = {i:char for i,char in enumerate(chars)}

encode = lambda st : [string_to_int[ch] for ch in st]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([77,  0, 41, 65,  1, 34, 75,  1, 39, 55, 51, 54, 55, 68, 69,  0,  0,  0,
        30, 70,  5, 69,  1, 64, 65,  1, 71, 69, 55, 20,  1, 64, 65,  1, 71, 69,
        55,  1, 51, 70,  1, 51, 62, 62,  8,  1, 41, 58, 55,  1, 53, 58, 59, 62,
        54, 68, 55, 64,  1, 73, 65, 64,  5, 70,  1, 62, 55, 70,  1, 63, 55,  1,
        69, 70, 65, 66,  1, 70, 55, 62, 62, 59, 64, 57,  1, 70, 51, 62, 55, 69,
         0, 65, 56,  1, 70, 58, 55,  1, 33, 51])


In [18]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [19]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size):
    context = x[:i+1]
    target  = y[i]
    print(f"when input is {context}, target is {target}")
    

when input is tensor([77]), target is 0
when input is tensor([77,  0]), target is 41
when input is tensor([77,  0, 41]), target is 65
when input is tensor([77,  0, 41, 65]), target is 1
when input is tensor([77,  0, 41, 65,  1]), target is 34
when input is tensor([77,  0, 41, 65,  1, 34]), target is 75
when input is tensor([77,  0, 41, 65,  1, 34, 75]), target is 1
when input is tensor([77,  0, 41, 65,  1, 34, 75,  1]), target is 39


In [ ]:

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y


In [ ]:
class BigramLanguageModel(nn.Module):   #creating the Bigram model class
    def __init__(self, vocab_size):
        super().__init__()   #inherit the nn.module parent class using the super keyword
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  #create 1D 78 dim vectors for each of our 78 chars 
        
    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index) #set logits as the tensor
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape          #B = batch_size, T= no of integers or block_size , C = vocab_size
            logits = logits.view(B*T, C)    #adjust the dim of the tensor as per loss function requirement
            targets = targets.view(B*T)     #adjust the target dim as per loss function requirement
            loss = F.cross_entropy(logits, targets) #calculate loss with target values and logits 
            
        return logits, loss 
    
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]  #attain the last character's embedded vector for all batches
            probs = F.softmax(logits, dim=-1) #normalise the logits along the last dim or the column dim
            index_next = torch.multinomial(probs, num_samples = 1) #draw one sample of the most probable char
            index = torch.cat((index, index_next), dim = 1) #concatenate that char and repeat
        return index

model = BigramLanguageModel(vocab_size)
    #m = model.to(device)
    
context = torch.zeros((1,1), dtype = torch.long)
generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

In [ ]:
optimiser = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    xb, yb = get_batch('train') #sample a batch of data
    
    logits, loss = model.forward(xb, yb) #call the .forward method with that sample
    optimiser.zero_grad(set_to_none = True)
    loss.backward()
    optimiser.step()
print(loss.item())    

In [ ]:
context = torch.zeros((1,1), dtype = torch.long)
generated_chars = decode(model.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)